# Vault Agent con WildFly local y AppRole

Esta demo ejecuta Vault Agent y WildFly como procesos locales. Vault Agent renderiza credenciales desde Vault mediante AppRole. Tras rotar el secreto, se vuelve a renderizar el fichero y se reinicia WildFly de forma controlada para que adopte la nueva configuracion.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next((directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("Could not find the persistent .env file")
load_dotenv(ENV_FILE)

VAULT_ADDR = os.environ['VAULT_ADDR']
WORKDIR = Path('/tmp/vault-jboss-demo')
VAULT_AUTH_PATH = 'approle'
VAULT_ROLE = 'jboss-local-demo'
VAULT_POLICY = 'jboss-local-read'
SECRET_PATH = 'secret/data/jboss/demo'
WILDFLY_VERSION = '36.0.1.Final'

os.environ.update({
    'WORKDIR': str(WORKDIR),
    'VAULT_AUTH_PATH': VAULT_AUTH_PATH,
    'VAULT_ROLE': VAULT_ROLE,
    'VAULT_POLICY': VAULT_POLICY,
    'SECRET_PATH': SECRET_PATH,
    'WILDFLY_VERSION': WILDFLY_VERSION,
})

for directory in ('agent', 'secrets', 'wildfly', 'logs'):
    (WORKDIR / directory).mkdir(parents=True, exist_ok=True)

print(f'Vault: {VAULT_ADDR}')
print(f'Workdir: {WORKDIR}')

Vault: https://vault.jose-merchan.sbx.hashidemos.io
Workdir: /tmp/vault-jboss-demo


In [3]:
%%bash
set -euo pipefail

export JAVA_HOME=/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home
export PATH="${JAVA_HOME}/bin:${PATH}"

command -v vault >/dev/null
command -v java >/dev/null
command -v curl >/dev/null
command -v unzip >/dev/null

vault status >/dev/null
java -version 2>&1 | head -n 1
echo 'Preflight local correcto: Java y Vault estan disponibles.'

openjdk version "21.0.11" 2026-04-21
Preflight local correcto: Java y Vault estan disponibles.


In [4]:
%%bash
set -euo pipefail

if ! vault auth list -format=json | jq -e 'has("approle/")' >/dev/null; then
  vault auth enable approle
fi

cat > "${WORKDIR}/jboss-local-policy.hcl" <<'EOF'
path "secret/data/jboss/demo" {
  capabilities = ["read"]
}
EOF

vault policy write "${VAULT_POLICY}" "${WORKDIR}/jboss-local-policy.hcl"
vault kv put secret/jboss/demo username=appuser password=version-1

vault write "auth/${VAULT_AUTH_PATH}/role/${VAULT_ROLE}" \
  token_policies="${VAULT_POLICY}" \
  token_ttl=1h \
  secret_id_ttl=24h

# Role_ID and Secret ID are typically provided via CI/CD or orquestration tools, but for local testing we can retrieve them directly from Vault.
vault read -field=role_id "auth/${VAULT_AUTH_PATH}/role/${VAULT_ROLE}/role-id" > "${WORKDIR}/agent/role_id"
vault write -field=secret_id -f "auth/${VAULT_AUTH_PATH}/role/${VAULT_ROLE}/secret-id" > "${WORKDIR}/agent/secret_id"
chmod 600 "${WORKDIR}/agent/role_id" "${WORKDIR}/agent/secret_id"

cat > "${WORKDIR}/agent/agent.hcl" <<EOF
exit_after_auth = true

vault {
  address = "${VAULT_ADDR}"
  ca_cert = "${VAULT_CACERT}"
  tls_server_name = "${VAULT_TLS_SERVER_NAME}"
}

auto_auth {
  method "approle" {
    mount_path = "auth/${VAULT_AUTH_PATH}"
    config = {
      role_id_file_path = "${WORKDIR}/agent/role_id"
      secret_id_file_path = "${WORKDIR}/agent/secret_id"
      remove_secret_id_file_after_reading = false
    }
  }

  sink "file" {
    config = { path = "${WORKDIR}/agent/token" }
  }
}

template {
  destination = "${WORKDIR}/secrets/datasource.env"
  contents = <<EOH
{{- with secret "${SECRET_PATH}" -}}
DB_USERNAME={{ .Data.data.username }}
DB_PASSWORD={{ .Data.data.password }}
{{- end }}
EOH
}
EOF

echo 'AppRole y plantilla de Vault Agent configurados.'

Success! Enabled approle auth method at: approle/
Success! Uploaded policy: jboss-local-read
===== Secret Path =====
secret/data/jboss/demo

======= Metadata =======
Key                Value
---                -----
created_time       2026-07-28T09:03:16.646709872Z
custom_metadata    <nil>
deletion_time      n/a
destroyed          false
version            1
Success! Data written to: auth/approle/role/jboss-local-demo
AppRole y plantilla de Vault Agent configurados.


In [5]:
%%bash
set -euo pipefail

export JAVA_HOME=/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home
export PATH="${JAVA_HOME}/bin:${PATH}"

WILDFLY_HOME="${WORKDIR}/wildfly/wildfly-${WILDFLY_VERSION}"
WILDFLY_ARCHIVE="${WORKDIR}/wildfly/wildfly-${WILDFLY_VERSION}.zip"
START_SCRIPT="${WORKDIR}/wildfly/start-wildfly.sh"

vault agent -config="${WORKDIR}/agent/agent.hcl"
test -s "${WORKDIR}/secrets/datasource.env"

if [ ! -x "${WILDFLY_HOME}/bin/standalone.sh" ]; then
  curl -fsSL -o "${WILDFLY_ARCHIVE}" "https://github.com/wildfly/wildfly/releases/download/${WILDFLY_VERSION}/wildfly-${WILDFLY_VERSION}.zip"
  unzip -q "${WILDFLY_ARCHIVE}" -d "${WORKDIR}/wildfly"
fi

cat > "${START_SCRIPT}" <<EOF
#!/bin/sh
set -eu
export JAVA_HOME="${JAVA_HOME}"
export PATH="\${JAVA_HOME}/bin:\${PATH}"
. "${WORKDIR}/secrets/datasource.env"
export JAVA_OPTS="\${JAVA_OPTS:-} -Ddemo.db.username=\${DB_USERNAME} -Ddemo.db.password=\${DB_PASSWORD}"

DEPLOYMENT_DIR="${WILDFLY_HOME}/standalone/deployments/vault-demo.war"
mkdir -p "\${DEPLOYMENT_DIR}/WEB-INF"
cat > "\${DEPLOYMENT_DIR}/WEB-INF/jboss-web.xml" <<'WEBEOF'
<?xml version="1.0" encoding="UTF-8"?>
<jboss-web xmlns="http://www.jboss.com/xml/ns/javaee" version="8.0">
  <context-root>/vault-demo</context-root>
</jboss-web>
WEBEOF
cat > "\${DEPLOYMENT_DIR}/index.html" <<HTML
<!doctype html>
<html lang="en">
<head><meta charset="utf-8"><title>Vault Agent Demo</title></head>
<body>
  <h1>Vault Agent + WildFly</h1>
  <p>Rendered username: <code>\${DB_USERNAME}</code></p>
  <p>Rendered secret: <code>\${DB_PASSWORD}</code></p>
</body>
</html>
HTML
touch "\${DEPLOYMENT_DIR}.dodeploy"

exec "${WILDFLY_HOME}/bin/standalone.sh" -b 127.0.0.1 \
  -Djboss.http.port=18080 \
  -Djboss.https.port=18443 \
  -Djboss.management.http.port=19990
EOF
chmod 700 "${START_SCRIPT}"

pkill -f "${WORKDIR}/wildfly/wildfly-" 2>/dev/null || true

for attempt in $(seq 1 30); do
  if ! lsof -nP -iTCP:18080 -sTCP:LISTEN >/dev/null 2>&1 && \
     ! lsof -nP -iTCP:18443 -sTCP:LISTEN >/dev/null 2>&1 && \
     ! lsof -nP -iTCP:19990 -sTCP:LISTEN >/dev/null 2>&1; then
    break
  fi
  sleep 1
done

: > "${WORKDIR}/logs/wildfly.log"
nohup "${START_SCRIPT}" > "${WORKDIR}/logs/wildfly.log" 2>&1 &
echo $! > "${WORKDIR}/wildfly/wildfly.pid"

for attempt in $(seq 1 60); do
  if grep -q 'WFLYSRV0025' "${WORKDIR}/logs/wildfly.log"; then
    break
  fi
  sleep 1
done

grep -q 'WFLYSRV0025' "${WORKDIR}/logs/wildfly.log"
echo 'WildFly iniciado en http://127.0.0.1:18080/vault-demo con credenciales renderizadas por Vault Agent.'

==> Vault Agent started! Log data will stream in below:

==> Vault Agent configuration:

           Api Address 1: http://bufconn
                     Cgo: disabled
               Log Level: 
                 Version: Vault v2.0.3+ent, built 2026-06-16T21:32:56Z
             Version Sha: 4580df18025099360823bbb4a8faaf7dba6d46f9



2026-07-28T11:03:18.275+0200 [INFO]  agent.sink.file: creating file sink
2026-07-28T11:03:18.275+0200 [INFO]  agent.sink.file: file sink configured: path=/tmp/vault-jboss-demo/agent/token mode=-rw-r----- owner=501 group=20
2026-07-28T11:03:18.277+0200 [INFO]  agent.auth.handler: starting auth handler
2026-07-28T11:03:18.277+0200 [INFO]  agent.sink.server: starting sink server
2026-07-28T11:03:18.278+0200 [INFO]  agent.exec.server: starting exec server
2026-07-28T11:03:18.278+0200 [INFO]  agent.exec.server: no env templates or exec config, exiting
2026-07-28T11:03:18.278+0200 [INFO]  agent.auth.handler: authenticating
2026-07-28T11:03:18.278+0200 [INFO]  agent.template.server: starting template server
2026-07-28T11:03:18.278+0200 [INFO]  agent: (runner) creating new runner (dry: false, once: false)
2026-07-28T11:03:18.279+0200 [INFO]  agent: (runner) creating watcher
2026-07-28T11:03:18.347+0200 [INFO]  agent.auth.handler: authentication successful, sending token to sinks
2026-07-28T11:

WildFly iniciado en http://127.0.0.1:18080/vault-demo con credenciales renderizadas por Vault Agent.


In [6]:
%%bash
set -euo pipefail

cat "${WORKDIR}/secrets/datasource.env"
echo
printf 'WildFly PID: '
cat "${WORKDIR}/wildfly/wildfly.pid"
tail -n 10 "${WORKDIR}/logs/wildfly.log"

DB_USERNAME=appuser
DB_PASSWORD=version-1

WildFly PID: 89339
11:03:47,508 INFO  [org.wildfly.extension.undertow] (MSC service thread 1-6) WFLYUT0006: Undertow HTTPS listener https listening on 127.0.0.1:18443
11:03:47,511 INFO  [org.jboss.as.connector.subsystems.datasources] (MSC service thread 1-2) WFLYJCA0001: Bound data source [java:jboss/datasources/ExampleDS]
11:03:47,524 INFO  [org.jboss.ws.common.management] (MSC service thread 1-8) JBWS022052: Starting JBossWS 7.3.1.Final (Apache CXF 4.0.6) 
11:03:47,735 INFO  [org.wildfly.extension.undertow] (ServerService Thread Pool -- 79) WFLYUT0021: Registered web context: '/vault-demo' for server 'default-server'
11:03:47,748 INFO  [org.jboss.as.server] (ServerService Thread Pool -- 45) WFLYSRV0010: Deployed "vault-demo.war" (runtime-name : "vault-demo.war")
11:03:47,758 INFO  [org.jboss.as.server] (Controller Boot Thread) WFLYSRV0212: Resuming server
11:03:47,760 INFO  [org.jboss.as] (Controller Boot Thread) WFLYSRV0060: Http management

In [7]:
%%bash
set -euo pipefail

export JAVA_HOME=/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home
export PATH="${JAVA_HOME}/bin:${PATH}"

vault kv put secret/jboss/demo username=appuser password=version-2
vault agent -config="${WORKDIR}/agent/agent.hcl"

echo 'Fichero renderizado tras la rotacion:'
cat "${WORKDIR}/secrets/datasource.env"

echo 'Reinicio controlado de WildFly para adoptar la nueva configuracion.'
pkill -f "${WORKDIR}/wildfly/wildfly-${WILDFLY_VERSION}" 2>/dev/null || true

for attempt in $(seq 1 30); do
  if ! lsof -nP -iTCP:18080 -sTCP:LISTEN >/dev/null 2>&1 && \
     ! lsof -nP -iTCP:18443 -sTCP:LISTEN >/dev/null 2>&1 && \
     ! lsof -nP -iTCP:19990 -sTCP:LISTEN >/dev/null 2>&1; then
    break
  fi
  sleep 1
done

: > "${WORKDIR}/logs/wildfly.log"
nohup "${WORKDIR}/wildfly/start-wildfly.sh" > "${WORKDIR}/logs/wildfly.log" 2>&1 &
echo $! > "${WORKDIR}/wildfly/wildfly.pid"

for attempt in $(seq 1 60); do
  if grep -q 'WFLYSRV0025' "${WORKDIR}/logs/wildfly.log"; then
    break
  fi
  sleep 1
done

grep -q 'WFLYSRV0025' "${WORKDIR}/logs/wildfly.log"
echo 'WildFly reiniciado con version-2.'

===== Secret Path =====
secret/data/jboss/demo

======= Metadata =======
Key                Value
---                -----
created_time       2026-07-28T09:05:17.131441706Z
custom_metadata    <nil>
deletion_time      n/a
destroyed          false
version            2
==> Vault Agent started! Log data will stream in below:

==> Vault Agent configuration:

           Api Address 1: http://bufconn
                     Cgo: disabled
               Log Level: 
                 Version: Vault v2.0.3+ent, built 2026-06-16T21:32:56Z
             Version Sha: 4580df18025099360823bbb4a8faaf7dba6d46f9



2026-07-28T11:05:17.283+0200 [INFO]  agent.sink.file: creating file sink
2026-07-28T11:05:17.284+0200 [INFO]  agent.sink.file: file sink configured: path=/tmp/vault-jboss-demo/agent/token mode=-rw-r----- owner=501 group=20
2026-07-28T11:05:17.285+0200 [INFO]  agent.auth.handler: starting auth handler
2026-07-28T11:05:17.285+0200 [INFO]  agent.exec.server: starting exec server
2026-07-28T11:05:17.285+0200 [INFO]  agent.exec.server: no env templates or exec config, exiting
2026-07-28T11:05:17.285+0200 [INFO]  agent.template.server: starting template server
2026-07-28T11:05:17.285+0200 [INFO]  agent.auth.handler: authenticating
2026-07-28T11:05:17.285+0200 [INFO]  agent.sink.server: starting sink server
2026-07-28T11:05:17.286+0200 [INFO]  agent: (runner) creating new runner (dry: false, once: false)
2026-07-28T11:05:17.288+0200 [INFO]  agent: (runner) creating watcher
2026-07-28T11:05:17.346+0200 [INFO]  agent.auth.handler: authentication successful, sending token to sinks
2026-07-28T11:

Fichero renderizado tras la rotacion:
DB_USERNAME=appuser
DB_PASSWORD=version-2
Reinicio controlado de WildFly para adoptar la nueva configuracion.
WildFly reiniciado con version-2.


# Vault agent exec

In [8]:
%%bash
set -euo pipefail

RELOAD_SCRIPT="${WORKDIR}/wildfly/reload-wildfly.sh"
AGENT_CONFIG="${WORKDIR}/agent/agent-reload.hcl"

cat > "${RELOAD_SCRIPT}" <<'EOF'
#!/bin/sh
set -eu

WORKDIR="$(CDPATH= cd -- "$(dirname "$0")/.." && pwd)"
LOCK_DIR="${WORKDIR}/wildfly/reload.lock"
PID_FILE="${WORKDIR}/wildfly/wildfly.pid"
START_SCRIPT="${WORKDIR}/wildfly/start-wildfly.sh"
LOG_FILE="${WORKDIR}/logs/wildfly.log"

if ! mkdir "${LOCK_DIR}" 2>/dev/null; then
  exit 0
fi
trap 'rmdir "${LOCK_DIR}"' EXIT

pkill -f "${WORKDIR}/wildfly/wildfly-" 2>/dev/null || true

for attempt in $(seq 1 60); do
  if ! lsof -nP -iTCP:18080 -sTCP:LISTEN >/dev/null 2>&1 && \
     ! lsof -nP -iTCP:18443 -sTCP:LISTEN >/dev/null 2>&1 && \
     ! lsof -nP -iTCP:19990 -sTCP:LISTEN >/dev/null 2>&1; then
    break
  fi
  sleep 1
done

: > "${LOG_FILE}"
nohup "${START_SCRIPT}" > "${LOG_FILE}" 2>&1 &
echo $! > "${PID_FILE}"

for attempt in $(seq 1 60); do
  grep -q 'WFLYSRV0025' "${LOG_FILE}" && break
  sleep 1
done

grep -q 'WFLYSRV0025' "${LOG_FILE}"
. "${WORKDIR}/secrets/datasource.env"
printf '%s\n' "${DB_PASSWORD}" > "${WORKDIR}/wildfly/reloaded-password"
EOF
chmod 700 "${RELOAD_SCRIPT}"

cat > "${AGENT_CONFIG}" <<EOF
exit_after_auth = false
pid_file = "${WORKDIR}/agent/vault-agent-reload.pid"

vault {
  address = "${VAULT_ADDR}"
  ca_cert = "${VAULT_CACERT}"
  tls_server_name = "${VAULT_TLS_SERVER_NAME}"
}

auto_auth {
  method "approle" {
    mount_path = "auth/${VAULT_AUTH_PATH}"
    config = {
      role_id_file_path = "${WORKDIR}/agent/role_id"
      secret_id_file_path = "${WORKDIR}/agent/secret_id"
      remove_secret_id_file_after_reading = false
    }
  }

  sink "file" {
    config = { path = "${WORKDIR}/agent/token" }
  }
}

template {
  destination = "${WORKDIR}/secrets/datasource.env"
  contents = <<EOH
{{- with secret "${SECRET_PATH}" -}}
DB_USERNAME={{ .Data.data.username }}
DB_PASSWORD={{ .Data.data.password }}
{{- end }}
EOH

  exec {
    command = ["/bin/sh", "${RELOAD_SCRIPT}"]
    timeout = "90s"
  }
}

template_config {
  static_secret_render_interval = "5s"
}
EOF

echo 'Vault Agent persistente y script de reload de WildFly configurados.'

Vault Agent persistente y script de reload de WildFly configurados.


In [9]:
%%bash
set -euo pipefail

AGENT_CONFIG="${WORKDIR}/agent/agent-reload.hcl"
AGENT_LOG="${WORKDIR}/logs/vault-agent-reload.log"
AGENT_PID_FILE="${WORKDIR}/agent/vault-agent-reload.pid"

if [ -f "${AGENT_PID_FILE}" ] && kill -0 "$(cat "${AGENT_PID_FILE}")" 2>/dev/null; then
  kill "$(cat "${AGENT_PID_FILE}")"
fi
for attempt in $(seq 1 15); do
  pgrep -f "${WORKDIR}/wildfly/reload-wildfly.sh" >/dev/null || break
  sleep 1
done
rm -rf "${WORKDIR}/wildfly/reload.lock"
rm -f "${WORKDIR}/wildfly/reloaded-password"
: > "${AGENT_LOG}"
nohup vault agent -config="${AGENT_CONFIG}" > "${AGENT_LOG}" 2>&1 &
echo $! > "${AGENT_PID_FILE}"

for attempt in $(seq 1 30); do
  grep -q 'authentication successful' "${AGENT_LOG}" && break
  sleep 1
done

grep -q 'authentication successful' "${AGENT_LOG}"
echo 'Vault Agent persistente autenticado.'

vault kv put secret/jboss/demo username=appuser password=version-5

for attempt in $(seq 1 90); do
  if [ -f "${WORKDIR}/wildfly/reloaded-password" ] && \
     grep -qx 'version-5' "${WORKDIR}/wildfly/reloaded-password" && \
     grep -q 'WFLYSRV0025' "${WORKDIR}/logs/wildfly.log"; then
    break
  fi
  sleep 1
done

grep -qx 'version-5' "${WORKDIR}/wildfly/reloaded-password"
grep -q 'WFLYSRV0025' "${WORKDIR}/logs/wildfly.log"
curl -fsS http://127.0.0.1:18080/vault-demo/ | grep -q 'Rendered secret: <code>version-5</code>'
echo 'Cambio detectado: WildFly y la pagina web se actualizaron automaticamente.'

Vault Agent persistente autenticado.
===== Secret Path =====
secret/data/jboss/demo

======= Metadata =======
Key                Value
---                -----
created_time       2026-07-28T09:05:44.799242219Z
custom_metadata    <nil>
deletion_time      n/a
destroyed          false
version            3
Cambio detectado: WildFly y la pagina web se actualizaron automaticamente.


# Clean up

In [10]:
%%bash
set -euo pipefail

if [ -f "${WORKDIR}/agent/vault-agent-reload.pid" ] && kill -0 "$(cat "${WORKDIR}/agent/vault-agent-reload.pid")" 2>/dev/null; then
  kill "$(cat "${WORKDIR}/agent/vault-agent-reload.pid")"
fi
pkill -f "${WORKDIR}/wildfly/wildfly-" 2>/dev/null || true

vault kv delete secret/jboss/demo
vault policy delete "${VAULT_POLICY}"
vault delete "auth/${VAULT_AUTH_PATH}/role/${VAULT_ROLE}"
rm -rf "${WORKDIR}"
echo 'Servicios locales, Vault Agent y recursos demo eliminados.'

Success! Data deleted (if it existed) at: secret/data/jboss/demo
Success! Deleted policy: jboss-local-read
Success! Data deleted (if it existed) at: auth/approle/role/jboss-local-demo
Servicios locales, Vault Agent y recursos demo eliminados.
